# TagTrail — deploy the poller to GitHub Actions

This notebook creates your **own** poller repo from the TagTrail template and sets its two secrets, so GitHub Actions polls your trackers every 15 minutes.

**Why not just fork?** GitHub disables scheduled workflows on forks. A repo made from a template is *not* a fork, so the cron runs normally.

**What you need:**

1. The output of `tagtrail-bootstrap` / `tagtrail-gfmt-auth` — i.e. your `poller.env`, or just the two lines `GOOGLE_DRIVE_REFRESH_TOKEN=...` and `GFMT_SECRETS_JSON_B64=...`. See [tagtrail.org/setup](https://tagtrail.org/setup).
2. A **fine-grained GitHub token**: <https://github.com/settings/personal-access-tokens/new>. On the target repo grant *Administration* (to create the repo), *Secrets*, and *Actions* = **Read and write**.

Nothing here is sent to a TagTrail server. The only network calls are straight to `api.github.com`. Run the cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install --quiet pynacl requests
print("ok")

## 2. Paste your bootstrap output

Paste the whole `poller.env` (or just the two `...=` lines) between the triple quotes below, then run the cell. It extracts the two secret values and never prints them.

In [ ]:
import re

POLLER_ENV = r"""
# <-- paste your poller.env / bootstrap output here, replacing this line -->
"""

def _extract(name: str, text: str) -> str | None:
    # Values (tokens, base64 blobs) never contain whitespace.
    m = re.search(rf"{name}\s*=\s*(\S+)", text)
    return m.group(1).strip().strip('\"') if m else None

SECRETS = {
    "GOOGLE_DRIVE_REFRESH_TOKEN": _extract("GOOGLE_DRIVE_REFRESH_TOKEN", POLLER_ENV),
    "GFMT_SECRETS_JSON_B64": _extract("GFMT_SECRETS_JSON_B64", POLLER_ENV),
}

missing = [k for k, v in SECRETS.items() if not v]
if missing:
    raise SystemExit(
        "Could not find: " + ", ".join(missing) +
        ".\nPaste the full bootstrap output (it has a GOOGLE_DRIVE_REFRESH_TOKEN= "
        "line and a GFMT_SECRETS_JSON_B64= line)."
    )

# Sanity-check the GFMT blob really is base64 JSON, without printing it.
import base64
try:
    decoded = base64.b64decode(SECRETS["GFMT_SECRETS_JSON_B64"]).decode("utf-8", "replace")
    assert decoded.lstrip().startswith("{")
except Exception as e:
    raise SystemExit(f"GFMT_SECRETS_JSON_B64 does not look like base64 JSON: {e}")

print("Found both secrets \u2713 (values hidden)")

## 3. Your GitHub token and repo name

The token is read with `getpass` so it isn't stored in the notebook. Choose the name for the new repo you want to create (it goes under your account).

In [ ]:
from getpass import getpass
import requests

GITHUB_TOKEN = getpass("GitHub fine-grained token: ").strip()

#@markdown Name for your new poller repo (created under your account):
NEW_REPO_NAME = "tagtrail-poller"  #@param {type:"string"}
#@markdown Make the repo private (recommended — it holds your credentials):
PRIVATE = True  #@param {type:"boolean"}
#@markdown Already created the repo yourself? Tick this to skip creation and just set secrets.
USE_EXISTING = False  #@param {type:"boolean"}

API = "https://api.github.com"
HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
TEMPLATE = "TagTrail/tagtrail-poller"

me = requests.get(f"{API}/user", headers=HEADERS)
if me.status_code != 200:
    raise SystemExit(f"Token rejected ({me.status_code}): {me.json().get('message')}")
OWNER = me.json()["login"]
REPO = NEW_REPO_NAME.strip()
print(f"Authenticated as {OWNER}. Target repo: {OWNER}/{REPO}")

## 4. Create the repo from the template

Skipped automatically if you ticked **USE_EXISTING**.

In [ ]:
if USE_EXISTING:
    print("Skipping creation; using existing repo.")
else:
    r = requests.post(
        f"{API}/repos/{TEMPLATE}/generate",
        headers=HEADERS,
        json={"owner": OWNER, "name": REPO, "private": PRIVATE,
              "description": "My TagTrail poller (GitHub Actions cron)"},
    )
    if r.status_code == 201:
        print(f"Created {OWNER}/{REPO} from template \u2713")
    elif r.status_code == 422 and "already exists" in r.text:
        print(f"{OWNER}/{REPO} already exists — continuing to set secrets.")
    else:
        raise SystemExit(
            f"Create failed ({r.status_code}): {r.json().get('message')}.\n"
            "If this is a permissions error, give the token 'Administration: "
            "Read and write', or create the repo yourself from "
            "https://github.com/TagTrail/tagtrail-poller/generate and tick USE_EXISTING."
        )

## 5. Encrypt and upload the secrets

Each value is sealed with the repo's public key (libsodium sealed box) before it leaves this notebook — exactly as GitHub's secrets API requires.

In [ ]:
import base64
from nacl import encoding, public

# A freshly created repo can take a moment before its Actions public key exists.
import time
pk = None
for attempt in range(6):
    resp = requests.get(f"{API}/repos/{OWNER}/{REPO}/actions/secrets/public-key", headers=HEADERS)
    if resp.status_code == 200:
        pk = resp.json()
        break
    time.sleep(2)
if pk is None:
    raise SystemExit(f"Could not read repo public key: {resp.status_code} {resp.text}")

def seal(public_key_b64: str, value: str) -> str:
    pub = public.PublicKey(public_key_b64.encode(), encoding.Base64Encoder())
    sealed = public.SealedBox(pub).encrypt(value.encode())
    return base64.b64encode(sealed).decode()

for name, value in SECRETS.items():
    body = {"encrypted_value": seal(pk["key"], value), "key_id": pk["key_id"]}
    put = requests.put(f"{API}/repos/{OWNER}/{REPO}/actions/secrets/{name}", headers=HEADERS, json=body)
    if put.status_code not in (201, 204):
        raise SystemExit(f"Failed to set {name}: {put.status_code} {put.text}")
    print(f"Set {name} \u2713")

## 6. Trigger the first run

In [ ]:
disp = requests.post(
    f"{API}/repos/{OWNER}/{REPO}/actions/workflows/poll.yml/dispatches",
    headers=HEADERS,
    json={"ref": "main"},
)
if disp.status_code == 204:
    print("First run started \u2713")
    print(f"Watch it: https://github.com/{OWNER}/{REPO}/actions")
else:
    print(f"Secrets are set, but couldn't auto-start the run ({disp.status_code}: {disp.json().get('message')}).")
    print(f"Start it manually: https://github.com/{OWNER}/{REPO}/actions → tagtrail-poll → Run workflow")

print("\nDone. After the run finishes, open https://tagtrail.org and sign in to see your map.")